# 🎙️ 會議記錄與摘要系統（Colab 免費 GPU 版）

上傳會議錄音，系統會自動：
1. **辨識每位講者**（說話者分離）
2. **產生逐字稿**：使用聯發科 **Breeze-ASR-25**，專為**台灣華語、中英夾雜**最佳化
3. **整理成專業會議記錄**：摘要、討論事項、決議、待辦事項（負責人／期限）、各人發言重點

錄音可以放在 **Google 雲端硬碟的任何資料夾**，逐字稿與會議記錄會**自動存到錄音所在的同一個資料夾**，關掉筆記本也不會消失。

---
### 📌 使用方式
| | 步驟 | 說明 |
|---|---|---|
| ⓪ | **開啟 GPU**（只需設定一次） | 上方選單「**執行階段 → 變更執行階段類型**」→ 選 **T4 GPU** → 儲存 |
| ① | 執行「步驟 1」 | 點儲存格左邊的 ▶️。授權 Google 雲端硬碟時，**請勾選全部權限**再按繼續。約 3～6 分鐘 |
| ② | 執行「步驟 2」 | 在出現的資料夾清單中點選錄音檔（或從電腦上傳） |
| ③ | 填寫「步驟 3」並執行 | 填會議名稱、講者人數等，開始處理。中途斷線可接續，不必重來 |
| ④ | （選用）執行「步驟 4」 | 聽各講者的聲音片段，填上真實姓名 |

> 💡 **每次重新開啟筆記本**都要先執行步驟 1（Colab 每次都是全新的環境）。<br>
> 💡 處理期間請保持此分頁開啟；閒置太久 Colab 可能會中斷連線。<br>
> 🔒 錄音會在 Google 的伺服器上處理並存在您的雲端硬碟，極機密會議請自行評估。<br>
> ⚠️ AI 產生的會議記錄僅供參考，正式發布前請人工覆核。<br>
> 🙏 本系統使用 Hugging Face 上的開源模型，作者與授權請見筆記本最下方「致謝與授權」。

In [ ]:
#@title 步驟 1：準備環境（每次開啟都要執行一次）
import os, sys, io, glob, time, base64, tarfile, subprocess, ctypes, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
_t0 = time.time()

# ---- 檢查 GPU ----
import torch
GPU = torch.cuda.is_available()
if GPU:
    print(f"✅ 已啟用 GPU：{torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 目前沒有 GPU！請到上方選單「執行階段 → 變更執行階段類型」選擇「T4 GPU」，再重新執行此步驟。")
    print("   （沒有 GPU 也能使用，但速度慢很多，且摘要改用較小的模型，品質較差）")

# ---- 連接 Google 雲端硬碟 ----
from google.colab import drive
USE_DRIVE = True
try:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    print("📁 已連接 Google 雲端硬碟")
except Exception as _e:
    USE_DRIVE = False
    DRIVE_ROOT = Path("/content/上傳的錄音")
    DRIVE_ROOT.mkdir(exist_ok=True)
    print(f"\n⚠️ 無法連接 Google 雲端硬碟（{_e}）")
    print("   常見原因：授權頁面沒有勾選全部權限，或彈出式視窗被瀏覽器封鎖。")
    print("   解決：重新執行此步驟，授權時點「全部選取」勾選所有權限再按「繼續」。")
    print("   👉 這次先改用 Colab 暫存空間繼續：錄音只能從電腦上傳，結果會自動下載到您的電腦，")
    print("      關閉筆記本後暫存空間會清除。\n")

# ---- 安裝套件 ----
print("📦 安裝套件中…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper", "ctranslate2>=4.9", "sherpa-onnx",
                "opencc-python-reimplemented", "python-docx"], check=True)

# ---- 解開系統程式碼 ----
APP_DIR = Path("/content/meeting_notes")
APP_DIR.mkdir(exist_ok=True)
_CODE = "H4sIAFfM/2oC/+19+XcTV5Zw/6y/orry9aHkyELyQogHkWHLcoYQBkhPf0d41GWpZGusrVUl27Tjc2wSwASDnWYJWwIkhBD2kE5wDJj/JXFJ8k+ZP+G7y6uqV6WSId3TmZkv1jlgqeot99133313e/fFN8Y3/vNefeJNQ88Ztd/8Qz4J/nT6m0j09nnf8Xky0ZPs+Y0y8Ztf4FM3Lb0G3f/m1/np2ayUrELJSCVfebWvN5FI9vbFexL9MCW9kd+sf/6//+jV6sZMplAuWJlMvHroH7b+N/X1dV7/8D3Z3/NKX39/X2/fJlj/Pf24/hO/5PqvVSrWWuWe9/5/6Se+zv/X+b/g//3Jvt7+RDLe/2pfYlOiZ53//0r4f7ZSzheG/1Hc/7n8v7f/lX7i/6+80tvbnyT+39PXv87/f4mPqqr2kZv2lSutm3ftexd/enLRnr/fPHOz+ZcH9mfvt+592Di72Lpx1L5/+8fpw1A4UihVKzVLqZiRfK1SUqq6NVIsDCni8V74GYls37Z/V2bnW/uUFD3QMpl8oWhkMtF4zTArxTFDi8ares0oW+JP5O13du7avV+uUzHjw4ZllMc09e09Ge+9GlPc5jcqaqmSM4qmGo1Gdm47sK1zA87bQPWcbulYOfKS0rr16eqVb1rLN1t3zwMa3jZyBf2AMapsrxnGn43ubfv3dff0//Rk9t9GCmbVqClFvTZsdI/1KKvHPrEfnLTnHzRnPm/N34d2Nq4s3m2d+Lrx6BFg0l6+17r17KcnxyPQBA8EQPRD574B8EyrpknoACCHGALdrAEE3VmrByHGKjt3/f6tHbtCW+NX0Jyq162KGlXcz0sKPoF2s9U6/l/P6dTYjnfe3vvugfDWxDtsrlC2NkvNvaTgA2gnX6zoVnITfMMHGfGTWt6+a9vb0Cw819qbxpfYbr8zqO3bDux4s3NxfIvlAYgodv/G3neVxvHvG3eu2de/tB/MwwzZ7x/hxxcOrzx9BlOAyN+x993MgTf37dq2c39o4+KdmAB4B+jJZCt1KBhVKjWlL+qQya3WV1+3po/Ys0dXL30GlFI9pJfLFctQTGO4BMSsW4VKubs3nlBeVnp3du+vGvoo0MuufYbZs8ewft8DIAKBNM4dQ7h2vrVtX2b/rjc6UIbzOpQwgEJrKn6Re45XyuUJWhBYddfb29dqGV6v3bJRGjJyuUJ52N8sYmv/m+/s3gnN0lRrIY27hXC+EnFnxlrffQhTpfxw9C9K684lwCQ+ePARIrdx88Lq409XD39gn7oCmN1xoPtATS+b+UqtZNQYbbCuEG17392z40CHkdG70GFV6+Wspbp8wx0Tdv3R+daNGWX37rehnzfeePf1n57MFYt6SQdCqCqNy3fsyw8ap262bp/D7qFYh87dN6EAFIsl6v5P40a5t7tvqLtQhkL1rAUrO/FK95/6MqOZUnx4uJ5HsLCtHQf+EEqv4h2iNrmpd3OfW/7Nd/f8C/y/bd/+zvW8MrT0YBMWc7N69lHj8vutuw+Auhv3vmWkNC9+0Lg/Dz/tux/DdkDdwPrK7N72f3et0YtXhOYfAXxJsZfnmrfvw9wSbgG1HrYBx42vriiwUAHDjdlziuXNvQnF3qwPDwMhKq/rWQMo5Zr96QmotTrzlb0wq+yoFPUhXPXO7GzftuNfdu3ZGTo/4h2C5YChMvbefD3zVngdeoM1/hWmbuO/8vxt735Lnj9oxN2G4qXRXKGm8f5mpg7U6kZMMSYKppWpjNLP6P8EAXtd/1vX/yT9b9OrPZvjm3tf7evr71vX/34l+l+1UDWKhbLx32P/6+1JvtIr9L++np5NSbL/vbJpXf/7hfQ/1HpufsjymF+4pWer0wuw7zdvPkPJeuE2i2wgqLVugpp0XGiFpAuCmle36jVQ9Bx1kARjkkrNiKQ5im81w/lm1oeqtUrWMN131kgNdiTY7rlp1NOyRd00DdNp230UU3QzV8haMSVfMIo5rmAdqqKsIMru0GGfHyoaLhDleql6COop5aoLV9UoZ7MRrh53arJtJBLZj3plchPQaySTzaKEQMXj78CfHTs01eyxxqsqyVDNB9cAb6wPNu5+vnr7DGNOPDnyOWgkra8+RUEFBKLmtzcUR6VEQWt+oXl9iV5ftJ+ctR+caj1ZtI8tyc2x8tCaPYaa+9PPWGhmpXvvPtDUDgB46srjL1YWTzTO319ZnLav/pXrYwck3q3OfQDTDuKTfeRL+973rCY1v59xW8dX15dXL11GPfbcMfvBbOPycXvhJECGUx6J7K1VhkGbN18vQ28OftNpkIZirBEMxpQ9lbIxGIlE/tmdqwj9r7xrWQYId1ljIIJKpMkq0gAKzPwA1qQ1wO3QA6Ock39axoTFpelnzsgrViWDRKCZRjEf5WbxUzOAIsuCQvhlGECkZmRrharFVXP1GpGt3GlRLw/X9WHDA7PuDMMcUIog2aXdcQ2iXoTUqAFser1oZfJ61qrUDqWwXPRFwZ5UHUBU6BWKxJ3fKLoKeJxXzu9YRAl8VA9QKJyux51eowqI10odBHhuwis3OMUw/rOJ6zdbMqyRSs6FGtcIt5BrB9rDpZZLewMAesihUK15gMMg/jyiRtsB9j4eSrWurroELzQtjWpwEKcVQYPpymX0eq5Q0dA2RZMVVbq3wkqPl3N6raYfYoiBbQHNK/l8qWoMK/aDYyuPH688Pdu4+sR+glaEUrV3Y6lP3ziuj20sVft+mL4BK7b19Hjz8BIwglHFPnevNfNwdeY0k0hvj2CF2Ha2lAMCSKvcOI6zu1wxLeBn9H0M/zdqtUqNfhbgf4QVv+fxSb63pwjYkdGidutZ0vWohl4T2uX+fVH8rQ5S2Urdgm49Zhqv1csawBJTsnqVODOUqNYtoY1kR4ysUEXiABy8i0jTCPjCaR6q5/NGTYOXMH/AVY0UvuARR+PZSvWQhqh/Sel2P65yNms//m515mHryXH7yBc/Ts8Av2rePGEf+ZptMlKVSKZYyY4C8C7fj++GB9B0Rjdr8BxZSSSDZgn4MTkVyZAq77zgqQfi4seaIMrhYmVILyr8kJ6MF6wRhfryyLaQFyWUgknNDfjo0du2Qp6aI0atqmfQjoC7iVnxF8orsP+hMonTGyftz9SEuV2yYEQH2laAmAQanfzcHbdZib+Tz6PYthef1GmRaaFPd1CHWlsfZANJhVZ5G191qkcUbmUk9TzVPqYY7rEZnk7TeS8Z4dCcJg1UzFEkknlz23624MAgawZQWKlaKBpaTU3DngRr7KcnMz89uQLU9NOTS7BJxuK/fW1QdVZ/VQzB0NwtghY//HVXvWfQVezFRd5emx9c420U97q7H0PTrRNfNO5921p42nyyxK/s48eAbpm2W59fXln8UFrxVYBWor+ImP6qQ1NoQiwaZc0bXjxfAH5ULBKk0ajSpfT0K1tTVIoetTFWfMo408cZObDUATEHzZe111Lpg/U+I5HoPlh/NZ/PD0bf017b0vYQiiIDgX/UhcQ1qnE9l8tUJVqCXnxTBOXiNaNa1IEXqzFsBiZEjXrPXuNnV+Rnv+VnMyAaATqwK6NoGjwUMWU1g3g2GqKTPRpN11ClUnTnawdtKEWYVLScymZ6EJAap08BJ1m9PI3Gux3v7tymJHtQjlKy9e27t+2HCsXCULY+BLt8HCg92cMyK7XdOPegedwxHWGdvYcOVGrZEaVx5nvcGri5Xhhl85vHzW+/Rsno4V9ADvI6Og5i0R2gFmoPSMZ+9BBrgkal2PeOtz7/RCmPFYBrdTMI8AchmLMffG1fvrk6faH1+bXVqwv2kVnmkSSQfWVPn+CG7fn7JMbdsO+el2jNEU2RG5vyE+R38m+zYDH3sGqHPGrievEdO3fvhr3Yjx2YLeQKKVFm34HdOzNv7H5n+7bd0fZtvs6NGxNZo2op7+zfhTua108VxSv8gnu2WSUhAwBCGQD/VvXsKAgBJsz4y0raeVM3jZr/7aDXYo4tc8RM/6NSAHZXBepiDCOd8UjIrFcYUj2Ii7B3ZYdwhQKC4vif5msmxzW4+m6LUdGlRmFvfZEaXnl5Tymi+pODfv3M3TcVYVNStNKJwReYhrC62aGfVzc4lc6n05S2Ta1o4HUdFrW0C8OeHdyD4dGaOzBu86H7L+ljed0EWS8zLhQlQd9Cb6LNKrjxih1HcpP9FrSibLWuhgwnwIKeP2FCKpEB0KQexQ6YM8YKWSPVBspaAq80sbDz1S0jQ0KX1IZwxr1gI9X6Wptw2Kzvoj+wA6BEA9oOqLQryyfsG4fRvL6RWJ997CkIyfbS7ZXFk81Li6BBkqnA4Zt73w1BMToJ8mq6dfeG/dGHgwruuN+cBfamTLbhZ2rjZPt4pxTmkbwBuCLmpDEFLB3dB27navTvmy+ikpgf/+z3/K/DuSP24LLgdVMvs+riLR20zRw70TizLA+5dfN84+h3bFTgnQsEkda90yDEkNOz+e0N9to4GDou7RzBxSgL1/I+kuXSWZC+ikWDBBpvcaMQXvgz6AQo4AnVHRRgQJDyHjUlwF9TRB41DkHHcgsoIXUn1+IRWAfladhICAj/sszmh31isfC77iRo15SDZe9pqq2B/dLb54rFji947Wb2ilLPbc6T0gUd+b3FHTTn5wje7ZVcJy+CLeDd5TzbNQGCPtou1gK0HUjX8Rz7+fBki3Vk9wKg14H373CfrAUFdiTqmimglhipk+ZIpZjzweY6pkP6LhXKGcdmAUSbSsR7n1con4dS/f5S0TBFEEg0PqYXCzlUUMKUPr0AsvG+ehk9MrT1amrAJOswAQzWWZ2737p1HVhf6/aJ5mdLrVsn+W3j1pnGtVmQGUHGDPBCWjdpQM3gc9aKBrD6OZVbM6jt2wsfNO5+sfrVFVmfR2bhcApAI6gCZJOJ+ZZ8DLYFtiLGFOBkRlHgxMwJbSqU1UjWs+yQlgOGA/NcsfSihFGnXY2aBc6OKthGpaRPaFQ0piSjbRJtIiIGbNaLpGLn4sKQ4gCfBX1tCCTTVHYoClJfzcoMHcqQsTKDU6b5MJbWanF6F1NqcaOcwz9iDGzHqiEn494GBX8VA+b+BiSblR9xMrf1kDigeEZZB5PGsAkj+VkzESQ5UHsEFaGpF9XTSZPg78rEFJNth8PmlKvz+pg6isBlZYsf+ogXPdSYOwaUvLK01Di72LhwGBWg2aPN5ev2zHfN2eXG5fftBx/BE9jd2PEAuhZv9K1jt+wTZ+3Z641zj+yjJ1tf3VhZnG7cuYYtzJ9rPXm/ceqL1rOP5PZBxWt+u9D49LLbv07zHIodP2rbsYLWraMnednBJqv6ZH99CCgvBFHQ3xQouD5sRAE7WL4cfO7nD2ImoYUIY25l+ZPVG+cw7unYUnN+efXqUXtpfvXYSdRqH91sXcAAGsaZkvxxekZ87flh+gZr/bWcIWxpjo4GUMoz6vVPZeOmYQljtobrFYZHz1F9SwYIX48pQzElr4pOJ6lg2hycUpny+b3bk0P9II1kWAbT/FabkuAHJBzJXaENFF7oFvwtxWkXgqniNkgjBEkuGpWaRzO3ZZiW1mYWsu/fWL14qfXoI3SYPMBwFycWcI5nmcQrjo4EKqQf179unP0YPSpEjJ4ECiKXfelT+/A33Khwr1z8oPX5J82n9ySBDEGBsWmw0GuwVCqluOMvqJWHtQTaZXXkALlMuVIr6UVt/z40FKG1KBFPJKNx2BtBPNUkm2y7wo8uB81FX9wS9vkhNJKZVsx1bKTQGg8zY+hAhbAeUsmYMgYiab5QhB01RUpeFBTMqKz2y7sVKwxuv5m8YJcZmAbN8E2cPNVieoLFw/VHR5E4WHZVCZwKn1mIpiVcN8CIoekr9tINnDVgEkc+Eyv356iZ/3Bt4nkaBCNMmsiQ/WKkYo3DujPJCBq+SbQBIPvdYIBAYjEFJBwhIKck4R4ImJGtUNdprjkIC8Mrr1H/GbMCM1vjPTiD1FKMKmhIPfcEOHPj8hUQVEBiQUPX8jNg5s1HF4F1N5fm5GUik7MjkEjDd4k7pgj27YxeZuFiM/YAfFFCLgRYUyrUkkDS29rE/18Nvlg4UlOCBT6vmdQaUwtFS1Wcftmj/LICa072AsPuMuk0P4VThVhyHrCRVxVLaxSt1eQtDHAaSzdHU6oHvY/5SAsKY4SDtIr9kNAF4ylVnQA7iV3Rg0AlfF3Va3rJIBVhUnX1A3UAqL23H7YMlO3NAmxvWcOT8UvoOE0mEonnWQBU4AhGdgR6yXGlTYnEVLASHrcr6MUMYzrFf7zZSrloBEokOSQStAyVcwWhoEAjQJaVuplBm7pg0hTKeeHkytNTq2cfoeng1pkfp+dAOGhdP2Z//7h1Y+nH6ZPsqSwW0HXBixg5K5EHbC/790UHBh2RzrfNUuRsp62W10Y9p6ts/ZQmkeK6typJb9WswXO361Z2xMi9Vc4bNZyKvSJGyR/y7e7QzNPtZ7dBzLPF0CTVHsi+UM5XkKY7tMvD8m2OfusC4AmoE2u3kyeOLKZ0dY0KhwlS/0BnGBiDUk/cuFffITshbtPkREF92b8v4pA+yoLpQc+8bgwLYYq6ClGEVHmHRJE+hlqspkEVVE5ggdPMYzdejEEynvAbtbE07yqR4GI040B/0JKmjbPa4zQZU8blHvAnlmdJcFyAza16fflR6PWhV6vUB1YJ9BIYCT9g55o8BI/p+dsP7lttDfrkF1EEZ9QNvHDYsZDgM7qlWcBlrSQ1RTquaXn7Z+P4Mqo1s8eUNBSzkoMKLNDmuQ8bl6ft6xdRWnQ0nuYH1/iVffwY1IICoNlAAWmHHCJJDv/PVMaESS+GG7hLIgGZ27+7wbq0kn58DIH4MeqpANgoEswQDgh1GNSloU0r4cfuGDQlwAi0F4AQ8FEZcxgMPm7b4fFhRDKf+MEWhRCnyCcdXSPJ+CgV0IYA+IfZQ3g3Kj3y9OFIeE5GjUOpol4ayunKxAA9R2VsAsRcHGMhh3syPUg6D6LRdA9qLJlde3a6nmrV9Uz/9jVQsVRH5SgWhssSPcH/gLfMsF5NJeP9/AOWeKqnJ0EKSSCKyKOVDz+0734M2y5ofaAmr547jUeOZh84JDIHuqi9MLvy9Nvm4aXmhSetm9PsvZZIpG5ZoXFKgo0QIiVbMNKMQ760SgPr3qxivMZziN1zOtZRYkYQ0t2ASZhR/M5ygi/IAd5oWJh2j7prMcF9BXvExzCp3fQOF+cWB6FtGzOZHoBsSBPH4sQLgDwF0t0u8DlANTBIlmV3VqMBDdwpqrycUsbb3iAsKRj+GhwMR+wwMC+sCUYV8xAt8SoxIfDenQ+KfKJ5dEvVGSaYiSya68tjRs3SvDAIfg2sslaoaoGYi3Td3yjNCpV3FHKMH3KjqDqboFx+49M5cFVIOk6YAoIbHK89FFEHRCsUl1StQPvcjr8Tt0EQ/gpl4AgkWvoL0VoKxvTBOvBaBvF1dfoh6oG3zrQe3beXP4KVQqGPj+zlOVBG0K21MGfPv48SBSyuK49a1+bolMr79vKnK4sfgvjBLTS/f+LXVoQI8x9mpRxpjyByAlxBdI1IwyBTDM6A89sJHOF4CX4ouU4mVRRDQMB0/NLkVv+zQfOFQWHyZEExv7VPdWYK3rhSvBN6aRmeXQg9qS7SaMUEo5nct9JyafOZOq0iTuLo59IwflaqHFNAGquQD0KtW/nuzbI33ZVCdQ4S0FTAgRpFjy58abejB8YQ5tsMB09UwJWXMfUxfJSGPXTQszjjUw1WDSj2LGr79tEAqrgg7k4423E2EQPvctvHXWarArw/BJw0DRIZNPz1e6RLKLJLfb2sqHF4qPrlJXSiEZbhFcz4uBqCY1R08+0IpGnK1UtVjUCJKXmsa2IEoW5mCwUxdr/8ZLpRP9ShRBi+cr7Bp2TMRCIBifXetD1/jsPTAXghaJA0jIzDH+jpmp8liXkLSMySwEDeld/rxbrjW+GmQUWCVbw6c6p55W5j9hw7pltfft74dEEVMEGzeMbSVEl6QpT4xHsSxq16tWhoEyzWTrgF01xxcDDid4WrivLD0VtK4yGaBpmduCIfW5jZVq2GqRRWAHEBWHxehE6G/miQkaZVuXc1MDlAtlYkQKA8MtyjjGEvPJKWCB9w86srjKFxWfD3VjRzoxhII1FGVY7ZuVSEbDDwVCXBNuoTDf2l0NhWBB1f9S9PanKrkvATvGdR5Clx2TtvB60LZ9GUmC9ZGWB32EQUrYrM9WWlyiEWh1WwQqGVjfGMEADZ+BW61nn4iEtG18uKWy+suIsKrEHN+kvhFEjgdKIX6IPNv/g+YE1cy3rETjfJfMRfw0hKRtBaJCWGnUKY/HQWr1fJYcq2EQGRg4BUQEHmN878C6sQjjCF/0XDiNTPdBpn/wqCtDiYPHvMPf2isjYcCQ90b4NCIkjs2A11b9cIXPeEoDDH/gpva649R/ZVjMQAgBIt87FSJadByd5NCQFaCVU89xWUiymbvKUCLbYpUHl1Eul/JDqQ6MlNDdCPkvzD5B8x+qGZyu+UJPof0BoGb3pzU2rkb2hOXT9euJ7/af387/+4/E/Jnniid1Nfb3Lz+gL9lZz/NeulEgmM/6ADwGuf/+3p6U8mvfxPvbj+8fv6+d9f6PzvyuMvOL2JSH7iZj0BKdd+4ElAIBk1F442Dy+hG/CLu3yQtHXz/OocGqLdzCGNU9fk5CEUGGw/vQC1f/5ZYe+EcOA8sIhLxcLueWAOnEBjUOSFzvTyQ+fwu/PSE+piQiKLrHnid42Ta8ViyT2gtv//7j+wC7MgsVtJXXl6lc/mrjw9yfh0z+YySptfnm9cWiSD00n7xAX0ltMR3pUnM1CYzuBSQ63bJzBgZfkEB5mEnxOWTx7f+hQT6VBAN0YIPb4CJTE+6JtjoAvjIb3T8P0WfgGwrs3Cl+Z3D1uP7nMkhdPrrU8bD7Ab+8k8lni43PxsCb40PvsEaATn//wpDIKZv9W4+n3j9EWXiPi8MZMSNLeyeNKev984Ob86fbVx+RaHMqEeTND89OSS6M+rP3+/9f7TxuXjsmKxOnefDm1hIOTK8iegveHR53PHQJCHblae3QNA0by3tGQvnGRssd7tNguo8rTvhZPNm18ThkE051w53rlq1TtZLY5SO21geNmdO8pkYWrjZHlKgRUAiHaOk15Ms9l+UHFM9hcZFeI8Eia/ufetN36n0KSjuE9FYHD2s6v2/RuN04/s+XOihmgETZaAfHv2Yz7QBhDC95WlM6tXj2Co2vKcfekSfOcDTHj+CDO6zTU/uPbj9EkMcfr2LM9F68KlxpcnaLIvRhJxRe6mceNw69kDbO7ZEfvabSjhuiM4/Iw8sQuA4datW0ABjfvzQN5Cl7pzaWXp6Mrj6dWlJwjus0+aZy/YRx4B9aGF9e7HSD0fAyfCo31OiMHxSDKutG6eat0+h5i+Noca2bGTq4+Rhls3H2Mf5+/bp2+53TTmFxCJgMEvp6nY8UgPDOLrJeRMs+fsI18DvUR646DlH2kt31hZOsEIcr7Mtx5ebT28A6QC3wGY1QsLtAZvYcPzHzLW8CfFFRLujkf6oLkHF6HF1WMLjcUj0Lt9dh6gjUSk5AkXI5NoWp9CFhh5/a0923b7qApml5cKk9ekWctOYc9/eQB9Ip94/NTjE7TuZG6BGKX1r7yt10ZzlfEy+oQu/hXID6aoeeRLpofm7avN+9OARCZLJhXKq3USWLXINHDku5XH57gkTAW/sj+8ah8/iV6l2XPNbxeIiRCFRF56SVlB6/eM4P7E9CPY5uMvlN6fnn66SbHnv7DvzcGaZ5eUoA0q3rx0j/yY0M1jhICmmokNoMSeON8WdbN4HPkUFeHpom6WP2HSQLMVzuAcUB18YXTAZL0EVbmEsgcgxrOc1+aY6IlLfM/LxiEreHgDyjSvLzEpNT6Yb904IS1JBgZWE46Z6MoDhgMvuUE+Q4rWxPOngDlyUZwVQUJ3GneuCST4ywiiYswia5ZJNfKewp0o7/lIGH66lAvfmXKV9yLvdXd3+/4hmC45u1VgWrkKLOsQAqcRH7mL6AfaAaCnj7CTkbFGbUprHSNxkftB9bmuLpmrdnWh44UqMaXIg118nwfLq4jHhbvX8hxa2p49bX1/nXoS6IVaP04vYFALEM6Nefv4sR+nP4p0Ky6I9vtHgNf5mCg5eU7ayx8zp2Va530Btw8HHbBvwS4DuFidvrCydEmwltlzAdQg4+5WGsenkaUS0QJjXL06i3Uv32Tqgp2sce4CbsXHl+3ZB8gwT19E2nCfE0dGAiMidakKv7iMkeZG7PPMpo/dagE7/PyyfeQIg4EHljl1G0HSfHBN2LOfLK18P88sg7sQB1BvHIa9hIt5TIRWO8h4wKth+I3HzwTLgK3iEWyXV7GvyCRF1U8FeJp7QAjknuDpP3i05uk/FJXCT9+7J/l8Sd2AZcrZ4ULO9DlyIx7obXvJKWKkBpzi2+pWheI5X6/Uduh1Uy/ufjtGTw9URo0ynkVoa8yqoPjnK0OZEzAyC/oA2TKnSWOgRHIhxzjrOR2lSIQ3jj/iBTOjj+kFSqyiRcNP74uOgyC39R9+hj8AVIx7z3CaB4bESeuJ84AQsjfSe9XbEx4Rx3FhmZJeTYlwMF8L5KQNH1LcGNOLIeMV4rQG6OYTrsXA6cU2f7s706TRZDCPo5jm3figUw/0kuPBMuhZSkl4ck40ZbLWhPx8x4E/PCcysLxGQC82OFytZ4r6IQxLlNr1cig+p/kxozZUMQ3ZMecEmuLa8x0xlNane8QQdT+l/Wxh8+5xZFRLC8CigAOhiEuJgd28qiGHC521Lus/bU5h/0nDjgvWdwTRW3DuGjFKVetQJqtnR5wl0tnRS4eWRYjWsFE2auhUKBmmiefMORTHwgVs+s3tPAqXr0X+Jq4UOjiPktGdKbDG2Y6yxApGMVqkCOMb0UERhbEW/SBjxgYxkgJFglI4Kfs8ePYpSU9oKGynj6hnAR4qQItq1SK/MTIhoOBCeRTdxwH3b4H8ewB1Wi2Uq3UrU0BfVtyqiGBRZgZeedoFeBYLTiBmBosGD8VxagpuxJ2yAvl/LICQQ3MxiJj69j8LAeAFcIBUgJ43poSURxQYv54xdfQrvSg+ccYQ5HrNSCXiPcheq5kqfH0Vp6dqWBzDWwXsFq1DqWQ80f8CrWJkMYEEWE4hjRgV033QFmaOBXJGFnELyEwnYjhZcXNErxrp5ODAYEwxRwtVdA9nMSJZDNtzjPEMAHHGs8C9LINpUfjX8JygQ40pjyzXGHfkBWkQ5DVLQkxSXp7SpATzk6TV7EilQAmh0onBtCpggl8qrFekDtUJbaKBaLxkRHiT1EVPom+znw04UVYuGaYn1VqliBE4qnnIhDFTELToZUBhg89UpxF7lTHjhb8qAzU16ONJMhBu5pcttCK3xrte27KRv0rZXTAPmz5spqD0/qgTAebbGaSIMU4H4z9XUgWWB9KDqQEOJLuYF6gIbYoQRSeiLK+mHU95neNzo1ODymTdCeNDmXGSY8ymVC/6zKrJqce8GaqXR02NYBjwegSSBRmL4s/CQOFaMQzKi9GJxfRgjP55obDFMnbK7cpiaFl5mcJHimUME8RenBjBQCQgdeGE8qkHyypnBYGCgaglHxAJORrSqQ19ee549H87ALysJN29Rgbg+Z2LyeCCAplOmCb6pdsmVFA44Axdx54dVSMRLxrzPXMDesJnTwpSSbvTPogDq1OMZLciCMMbUKAcIoBJRBA9HtilvGYlzcLz03VMC0NJ2ZO+s4Cww4Koq+HpRCCIQf5f2aiIE79Z+MWxJxQEjePyB/92Y5BvNBDFbWYK5UAyqQDBmYZR9kfjl/hgL2V4KgBiMEfToKLFX34tCvTvJF/ykV4pPlyr1KtaMuqkOMBmg0dAjbIz7155/3lIKOIMAGa/XBkvZ8o6sEFtqJI7JPicF8LpjoR5nSEPC6OOb33WuPIFKL2hWj8owKwcohJ7/r69cIOValRQ6ey5ZDSZRfGST57YM982Fu/4RccC4A8BJIxpKlsbROSTaWTF23RhwF9oZfF9NUrBxAXMnJVwjh3xYWUcNsb0lQXvUxVA/oA4gVqWpgiTcMEMdR3s0tL/3nWwPDiZjPUkpl6LHjS70lhn8DX4hu8x656RjU7JKJ8sew1yn8RJnOhjvXxIK1P0suJQXtkjQvew75SYtRL8zJQK5brVxndhOzAsncNdvZCcVHsAbfuWQwJySDBt2yFYKoi2afLgNO8eRZXg4m0+oidHyKLZ5dFHaEl49BCNe7OoG7B9D+YfiAUK2DMXpTkmdivFGUm7i+An9SIuc4+j0euonCHUlCLOaWFiHafEKIfhz6hObignXYAUGEjllS0pWXyXM+Z3Kcl4vxTbV8uKzAQxZ+9VA1Z/3HDlH9hDSMReFTgeQS9varEOYESDesOIbuItK8HHboRxeJ5CEUEnRcHiXoIzHBrIiy+iL5CsSjQbDOXF6s+N4l07DDccaMyUIh13cphsIaZQTjGjXC+xREY4xswO/lZHoK5AYLyU69cwQmlq4ySSA9WITh0sT1an1DjBDjpIND5iTOQKw3RMPGiNGnH4c67NVuULJMurPi9osEugFK0AmyEdTdG8F3SYPhqKlfTIIBMQyK2yVyqO6iY8C5U0CynAUznldcCEnKp620BKWjS+xVWNYga0ZH8i0Q6QQ0kDob16sceCLH5W9HF7FDLHG75AEDLRirNH5mFgC8Ilh+64H6c/At4CGBbYjK65zjGtCUwfr3ba8Jgdss/B7/U6jlxA5pn4+2DZ5WQEVjDSkJ3isheHAltfjfpP4cquImeqAd6UB3NKgjzFkr87t8AWuT3cOYl8ioZezuAvTVaBYkovxvOJsjqeZQkRILyGXa4KZSlFGO/97D1kgzp5/tEteemv7InCHWLuWPP6Em4eTr6OkJOKjavXGnc/b0dMvzdhzxkNzHGe8I/ugm9uNj5ADyy5hbFL9nNx9hz78gMhtSwdlQUXHgKIPpMbYHVs4GmEsUan2HWkPl+TzaueJ2L+FqaNnITZmGI3teuba5y+yK4A3DDpli0mJ2k6hKAxQiF6GmmqIAggq2Aagy+ED/qtutqThxpX9vPv9wKJrjb573/84x+11wZKwoMYRannPfgHj/+P0CtJ/nIOE3UQ3VaW5lQf4EJ2axPU8LmA1h1bJ4HHDzlpMXQ2yq/VCIWhYBVxD8HaHIJLT1RSGcT+LWJWK+OmT+TIq+/BLE2BlOSosSTv4e9q1hqIJ34H31lxpfBdWIJVlMYyTjS73ySRB+HnJWWSugcWJFG08HhNu45TjniIdCtdXcKN4znMRAPSS/aU80t3mBswTHoDjXIDBlNcuw0EtUGuBtQOfCxYrVjJkgGxQ9WVxcf24t1gJVhthVqHGjxEqrHtLUUkR5FjWyidKrrwbxOvePRF68bRxtVF9OlJY7yAQUXcrxMbXXMToEdFV84Swxh93OV4EqaUlafoWOTa0hI2nbnrkqYMhO/3RFwGzLQ4PkkM3vt5/8zK8omgQzUyiRRE7q/1+N/1+N//5fG/eP93oqc3/mofTMArm9bjf38l8b/GBGqO/233v77S1+Pd/518heJ/exN96/G/v1D8rz13n6LALh74wwE6vYT/O2Fk8PXfKrXc3xS6K67uqWQnnII7K9k6nhfz3sRRfecj7E62rZ2ZbbvfemNPZu+2fdve2Ldt75tS6cpEqRj3Yn7/VJbemSN6zci5N9GCVLbvje07KsVKLfIzYn0j4lYWa8IK83u4IqiQ8GQtL8z5ASq+9wyU0oA75GDZ9YZ0UijCzexRv7QPAJu1tQFmT55kpS7EuF3PgOLrwW9IweT/rloN+jQA7h9qjLKI44ChU9+QnTc01hA/EIwiGkCpQCh0GuKTKuXa9ASSjztrOP5J6uqS4Ojq+nF6+o9t0/ZHrPE8R1VA1PcEfdceSK0gEFOSYobuZSBYzkDorAhx7VAFbd3OM2F8oiwQ8AZgO1Q0zLS6h1IBijtfTCuer5StOCrpZLZYvtd6fAUDwB9/tHr7jOorRC6tFKwNLZl0Go8bRQN7i9f21uK116GYiaketT+VNXV8wNBNa5tZ0PHoYlvbnh+Iegf8aOqbHASv0EU1zo8e+Ucv/jjAGplkavUPEhv07H3mC4yRy/29g5F6yyLriNeG8eIAh5NoiYnk6zElMdG7Df/vf91HAwC/M8kYmlGrl020usW8+6rEgB37JeYv8q1A1MWrxYLl+CPQZSQcD5zJyOc0wvqBHHR4k4dei4vuyebnXcbxR9LiA+bFWnyoUkR7T0H5ndKDnoqk44vAQAMk1QmtlHPSCLp3KskU69G0z9RfyvFo2M7v2QvYETo+UgA1vaBsYZcnGfulpJWUHAQfpguD8Zrsvnbs+WVerSZaGzX1PTVg9xXKvWQ27tQrmcPdvkRXazdOXhWjWKQesk4dmtis61z2t0Y+KGhIzDB+HQyzq5LXCF1SRhydCSXMOQb0MPBa92RPbGoAb1nJSj0RFNFwKyqiwGHbXK69Q3S5JoNWbqzY3mQZlgROqz5BFmtfGmOo0N64w7mQHC2KY8RyZI2mCugAKZopbDekMvMCXPC7C8MjlvJGrZBTtmWzsL6dNE/BjRIoFBoOrCnsKRw/LhKBoQ8bGsHR2R6NGEQHVhy/aDWchjXLOpl51M7GQo9PUAXMTThc06sjJt3jAZCns2Q2ywp6hSdRf3LFUKN/XqGESYmBNa2UhLB62SEif+94d5g58FwrJ5Ry+EfbLSIY0VIoSw9LbG10KPrftZcmk7G+KbyVSIt3RYGw5WgI9Ir7ASgaYxwrB5gIOsBlb49DciO847hle6SLibq6mBnGuNUUZv6ibzGltz31jOg5pSTbUTISpzP9uO1gUtg28TW+Y9eeA7v2SYmhcH4kNKCbubtrkO9mKpYDFOiRiDMsd540WiApWB4gI2wHXmFYOCDPoOtrWeXWo2sCcjD3cjr+4/QM1On6e6DZUy8NGbU2aALNh4NUdDz2P6PvaIB2PK5GQgVme+AsMW3iJO1vnQXKF9zywsgOBOWgQKg65CbEr59DO8+Lt6lKIEiIifhkA1cyAOA8KRhkYImd1EJXdO1nS0VUqyfQa0h8mNx1T1BQfbX9ZQcQNiewc+f/qOQGD+JEji4KkMd6/od1+++vOv9Dz6vxRO+rr25Ort///mux/2bxqMc/zvz7vPwPfclEr5f/oZ/uf0/09a/bf38h++8blcowqFniwA/dU7F6+/zqxVONb2eaN09gkoALZ5sLR/nOcDyiefe8Pftg9dLD5u37zWt3mp9dce8XaX11ozXzFJ2yZ76n+ImfazamkLb27A+YLJNawV26WBhyrbzwM5jegd0ZMcWx9sYUN8FJBzswJuwKtQXz5eBOvtIBNwtkJoMp3DOZ4LXddIk2ENQwdsp3cpuW+Gol5LTMsfbEh9wwXjEkGqYsYthUVeqiXBnvkMyNM2QOG5gW0wOk7Thlp1deKrxoIOo4ZEycmA8fJfyyO4LXLZXbovSQdaUK3wK6ExvN/LgTbUMrUrNomwKwpKblpNNu+r6aovww/40ySYBNKcpkdaCPAjYUBVMWPHrY/O706swpZRJa3LhR2ZSYUuzZo/Tzd/CLspIpzS//wvE3RjmXQvUkX6ybI+JAjAil1/MGxURpZojJWZgCHa1nQ/rgwY0DXa+pW7a+d9AcfHkDKD0ZDCN2LUIZNZoe2JQY9AWniNugdEvH6wxERj46jUi9Idl793DQ6uQgIhiolHLghBxagtlcKLqJ13Hj45nVwx/ikd/lSyvfn7CX8ZIUcVT7y9OYJ0McoHcDePF4KnYsA+O70iWOtjLGDGgXehwPxUxlZPDiuLxVD5Fjzo3kptQoLI014p5FbLQczNyOEj6SzQk0V99/KhgWj/vWGXjLQV72whyeqZ4+wdDZ15el4QJguUItZMxoJ0Eflhxq5NEEwpoWAUciwjB0GuPjtQJeBwOKiOYGOppeEOekii2pAzT4mKJ6Gis8Aw0QLbN4HCPKV1YgTtQBBzvSsaP2iEnMWp/DpI1JrCtCMam8Jsa8UXFV14xLTnFrwlJ9YAu/sess+3nNmbWOzbErK9ic875NdWcCcZEeo+sq1uwbKzp2cDQzCZpykRZWWw5gipdyfthFCwGIJahlW7pTtgOYvo5kSL1zblBFrCFePkRd/jXEIYLOaRWk4cEBia5jLul2IO/Y2jTtHPiqklmCArFfbCwx5QVmhpwbTmB61D0Q5oSItg1X7I9/F4fiZN6CQ8U6leVwbl9Z9/oqkBpy9RpFDA6EJOOk5JfwzsvI6aXMLIm9yHR3ITowX2JbLxQw+aF7Z7p0LWFGytDu45R4lIRFn0PuoWDO3yqnrnZ3EEe+8w55uHnP3SvOQlOf//TkktMwdqPYx49BudWrd1fPnbC/PCHx1OwosnOTzyCk1ppQZ2mKhgd8of95Ije5sVg4kw0YEPPxehnkvlFYgqZZKA9nKqMpfw5YJCAnDjnalp6dH2R0dgMERDBHDvnPK6e+VBiX6ORuW15IUVOqm+zaGPNdwNth03fXgnyzC1V1j2GYWnhNTDrATMp/JAOXGDZB0hl8EZGy3jbjy+dO1kDnFaeXoO0Ha6blWlJUPRIj4q2ri0vRhjaIF91Mjg4oYzSVozH4ArNJgaXATkt4HzyANjY1FZ4wG+Q4zEXgCTlzzbNfgFgn5wNDce/GYVyndB2ie0sQxl2fWF6dWfxxek5QKBcg+udUNP6LHD3EOHssOwvltRWWTdoHbCBTnguaff92IIcTXuR65qa4TbVPcYEV1zqe+ax55oq/tZMBN5Bz7JtufCOkC7gHmTmEHE3KER05OlHcdzNfUMhWfBftOVkgLk5uwDuZfrh4bQNRMDaYUjZghoYNzME20NV7s3zjHmB5w5QEN1GX2z+aiYMbGU+B70aDqPzGvc1AuHRwKa911tzLTZ/CTRg5STTmLPWU+BuiYpR/uHzEM+bzuTccPjqk/KEhdOCEQ4dDTgsAQ6frQV0LfAfj/hQHMru3vLD8unr2kRIaES1w6q07ITt66FRpn5D5w5Rf2F1DJo8JUZRFbjzycQQvGZaX4urMM/vISbFHXjwFKBA51GG9Hbm5+v7N1Zkl+/rX7l6wJj8WlBtgj+Erz6Ue+RZ1hFG6Gd3NbhJkLP955fQnSvviwnxed+7wkRF4vrJ4ovXkuP9u9bnmXz9Qkj89/bQXt+/VhfN+BtIp7bo3Mi/1auDgp4NsomQ+akWkyttdNCTTuu+8zFqZ1sWomZjlWkzPqj9J+nOIQUAs1SEAn7PROjvl6RmFlTQ8RSoraY6G5ipmuMqYSB0q965KkVmZpReK/nOgsNZGp5RJVzwbi4qALmfrEa04u0/7sv9h/uufFueV1vTHdEPxxUnuZgoJZPFp6+as1HoA6Sgr4HUBz1u5Ht1IDNxVFIhxs+RHBy7bBN88Splhu798R0I+RIjgOzFeh1p7KtbrlXo5x1dj5NXV6QugGPN0rB7+oPHwL0JKnDmFh6qO0zHv2QcwHJZoOD0o7rDAFVho5G2sVwws5xdW8h0Fk7bs+pK4kQvIGiD3eXJFLrBT+1MfvEgai/aj/Gs5XOm6JIeBv9Cpfbd4p0P7dJmjMZ7pALO42FOW+EGjqJRzZmqzdLGclIpv9dr95vUlTsgnTrKcO4ZXiWFW2PuNv9xtnMF8o/apKySvzjXOfb+y/Ay391snG+fvt249kMR3sp6+tfeQNYL2ioJZLeqHvMxpdGeEEzcMs8Hv21M/ORe4uDxbusklSLqdzws7rfjuK/Odqddcj71z+RZObb3DtPpnE8/re8FZdRH0RNeXDSQxNMefOWJCpLaYED5t2h3pFnJY8XiaUCREnIMn9hdPGb1SHigvwUWgJe8yGexb8mkzcjUH3RzWak7headZ+8jXPNUer8EeiMuER7vCUxHsmh7YnMBruv3XmROuCyYReXuOPrxZUNhoRZN8f2iMHmFgjYgDrntXNjLVRrmgrzVnaNs8gkjrA0MYDqVbRgqKOwu7ZpB27hMWYaEBM2cVOM0LhFLG1AwncY9P+W3byMTFQm2ESFcukdrii/Vz1Rfu16fBkKLgFff/wqurpl6Iy0hECVAQi3MfxZQARyGN2RnpQKig00mXwFsL/yYRxVOYZUFlDd0es6x4Pa1dcE1BIuiyIA+EAzmpaVU5A4NInYTJq3CbVN8TEgI8M0BsqRraKIbV11Tttd8ezEWFlEAshVPL4LVWtPCNMlLUGDAZTgYWjYZhDnokJ4DgE6UBgCftxKIlooMB6ennyOC+iq6cAvIcapyX/goT3C4wrSFZbHt351vvZHb9AXMRa2q8VKUY7XipT6e/4/oY/6720V9dz9LffFF8qQwP899q3eQaxlApeL85PC1xe6UKt6eXKPNVvHe4KvoT/YyO0e7N7jeUTrZjEKfh5SlB6ZCTk1y+g7m7p+dbN54qwpMpeyZR9Hr8KWjRrvVf7GfhvjyMuaw4+cDEbejtVxDyzUeYlVC+N1tKwlGoHhov5PBGPkxxMO73cWEPjqESv0f9r7PjOeet2EzYsVfjO+WkF5ghFOZXExGhbuMBdyQBauQGqKp/JP6S7qgw2b3z3V8E6TBTHMLdajz+5oG3d2sB6FGgKtB9UuPx/dQAh/wm+zBZwSGgyNR4fDd90QBF1khKTSYSv5M3HLmdeGUIpn3MoMmJ4xH/8rABOwmlJEiplAxKDVStVzNDVpkg2A4MtVLWcgaLTyBApNQf7hxVOBWA/fV1tTNUyUR1og0sbhsxlS0WsqPO0s4M8OvMcEVzJlHY0YMt1Iw8Xjm+Joj/eeXMB4JPc6b8NcDsCQNT6uSFYA3Wx2PRdbPjJA9VJujd77dXJrQ0lKG/EoJibWDE/PQzuHYuRpkEYjJIgwFIfIOQ1nTNKIr1XGW69ztpnXBI3KiqXlAwKdjOKkL3U2MWr1mW+QleA9aWZLEm9SwAAvlXdDwQCTrfiQEIfSENxNzl0Io56Besckqq06pmLpELpU6QoDDWngqQJNvWRFteH5JbsDfe6HIoyNSIt/hEXUAm5jJRqi7rAQST/gdC/LhR0+SNUMju7+wnlRIZoTEQGlkA88pp3QCAvLrFrOp4MB9jiTdQqOfAS9lE76s9Qxu2irsZ6V5In2nAmNqyEettVUOHJZ0BqVStkGRC5MCi0m3xEmJ88mmQeOhpkGC0u1vfwVV7HQTGvdJcGEQmq8JFEBPkGUyhVKRGzXo+X5hw0I7gu5t4x6RKmqqAjAAiAqvyqt/dUPW8DB3XJh89UH6YX/xpcZ5bwaTmwEtPfkjNBaW5ahTPEjjtdj660IaJU9+6mJhE8EPx4dsq6uW/YbNw95kqXyaTYgIBkkhrKho67l53CU3cnXF3wXsye86VK0QWIPlQj68Hh8Lb992/Y69z+Km8foacxQNi+3Ch3F008tbA5urEhq1AX4eVSW4eOWQuCstmaKssDIleBQ/LyuxrjC6dTatlY1wdjAQOLI2tGdtTdSSaMb93pcPScFl7NbrGddo+6QbF7ecyFyCsLbnCmIOfqp7DOP2BTdUJBXf7fxrSs6MonZdzAy8Zm/P9xqv/JBhQcqjf6En80wtl/Rmq1HJGrbum5wp1ExvfsFVI5iCGNk4fA361ZWirNA1VnoYtQ7WtL9QBm0zxImoymYZGsaAniZyzbItbWTzBZdgoBwW2bARMbFWjoeFhrvQXcrey7z3O0HoQ7fpn/bP+Wf+sf9Y//8s+/w8ygnWQAMgAAA=="
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_CODE)), mode="r:gz").extractall(APP_DIR)
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

# ---- 設定 ----
MODELS = Path("/content/models")
os.environ.update(
    MN_MODELS_DIR=str(MODELS), MN_DATA_DIR="/content/data", MN_THREADS=str(os.cpu_count() or 2),
    MN_ASR_DEVICE="cuda" if GPU else "cpu", MN_ASR_COMPUTE="float16" if GPU else "int8",
    MN_LLM_BACKEND="transformers",
    MN_LLM_HF_ID="Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-1.7B",
)

# ---- 下載模型（從 Hugging Face，約 3.5 GB）----
print("⬇️ 下載語音辨識與講者分離模型…")
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download("SoybeanMilk/faster-whisper-Breeze-ASR-25", local_dir=MODELS / "breeze-asr-25-ct2",
                  allow_patterns=["*.json", "model.bin"])
def _get(repo, fn, dest):
    dest = MODELS / dest
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(hf_hub_download(repo, fn), dest)
_get("csukuangfj/sherpa-onnx-pyannote-segmentation-3-0", "model.onnx", "diar/segmentation.onnx")
_get("csukuangfj/speaker-embedding-models",
     "3dspeaker_speech_eres2netv2_sv_zh-cn_16k-common.onnx", "diar/embedding.onnx")
_get("csukuangfj/sherpa-onnx-punct-ct-transformer-zh-en-vocab272727-2024-04-12",
     "model.onnx", "punct/model.onnx")

from app import colab as mn
from app import pipeline as _pl
SELECTED = None

# ---- 語音辨識引擎需要 CUDA 12 的 cuBLAS；新版 Colab 只有 CUDA 13，需另外安裝 ----
if GPU and not _pl.preload_cuda12():
    print("📦 安裝 GPU 語音辨識所需的 CUDA 12 函式庫（約 1～2 分鐘）…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12"], check=True)
    import importlib, site
    importlib.invalidate_caches()
    if not _pl.preload_cuda12():
        print("⚠️ CUDA 12 函式庫載入失敗，語音辨識將使用 CPU")

# ---- 確認語音辨識真的能用 GPU（順便預先載入模型）----
print("🔍 載入語音辨識模型並測試 GPU…")
_dev = _pl.asr_selftest()
if _dev == "cuda":
    print("✅ 語音辨識使用 GPU ⚡")
else:
    print("⚠️ 語音辨識目前使用 CPU，速度會慢很多。")
    print("   若已選擇 T4 GPU 仍出現此訊息，請將上方的警告訊息截圖回報。")
print(f"\n🎉 準備完成！（{int(time.time() - _t0)} 秒）請繼續執行「步驟 2」選擇錄音檔。")

In [ ]:
#@title 步驟 2：選擇錄音檔
#@markdown **從雲端硬碟選擇**：執行後下方會出現雲端硬碟的資料夾，點資料夾進入、點錄音檔選取（🎵 是錄音檔，✅ 表示已處理過）。
#@markdown 逐字稿與會議記錄會**存到錄音所在的同一個資料夾**。
#@markdown
#@markdown **從電腦上傳**：上傳的錄音與結果會存到下面指定的雲端硬碟資料夾（不存在會自動建立）。
錄音來源 = "從雲端硬碟選擇"  #@param ["從雲端硬碟選擇", "從電腦上傳"]
上傳存放資料夾 = "會議記錄"  #@param {type:"string"}

import shutil
from IPython.display import display
from google.colab import files

try:
    mn
except NameError:
    raise SystemExit("⚠️ 請先執行「步驟 1」")

def _pick(p):
    global SELECTED
    SELECTED = p

if 錄音來源 == "從雲端硬碟選擇" and USE_DRIVE:
    _browser = mn.FileBrowser(str(DRIVE_ROOT), start=str(SELECTED.parent) if SELECTED else None,
                              on_select=_pick)
    display(_browser.box)
else:
    if not USE_DRIVE and 錄音來源 == "從雲端硬碟選擇":
        print("⚠️ 未連接雲端硬碟，改為從電腦上傳。")
    _dest = DRIVE_ROOT / 上傳存放資料夾.strip().strip("/") if USE_DRIVE else DRIVE_ROOT
    _dest.mkdir(parents=True, exist_ok=True)
    print("請點下方「選擇檔案」上傳錄音（大檔案上傳需要一些時間）：")
    _up = files.upload()
    if not _up:
        raise SystemExit("沒有選擇檔案")
    _name = next(iter(_up))
    SELECTED = _dest / _name
    shutil.move(_name, SELECTED)
    print(f"✅ 已上傳並存到：{str(SELECTED).replace(str(DRIVE_ROOT), '我的雲端硬碟', 1)}")
    print("請繼續執行「步驟 3」。")

In [ ]:
#@title 步驟 3：產生逐字稿與會議記錄
#@markdown **會議資訊**（可留空，會議名稱預設為錄音檔名；之後也能在 Word 檔中修改）
會議名稱 = ""  #@param {type:"string"}
會議日期 = ""  #@param {type:"date"}
地點 = ""  #@param {type:"string"}
主席 = ""  #@param {type:"string"}
#@markdown **辨識設定**（填入正確的講者人數，分辨講者會更準確）
講者人數 = "自動判斷"  #@param ["自動判斷", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
專有名詞 = ""  #@param {type:"string"}
#@markdown ↑ 人名、公司、產品、術語，用逗號分隔，例如：聯發科, Kubernetes, 王小明
產生會議記錄 = True  #@param {type:"boolean"}
下載Word到電腦 = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **中斷後接續**：處理中途斷線時，重新執行步驟 1 → 步驟 2 選同一個錄音（標示 ⏸️處理到一半）→ 步驟 3，
#@markdown 會**自動從上次的進度繼續**（講者分離完成時、語音辨識每 20 秒、每段摘要完成時，都會把進度存到雲端硬碟）。
#@markdown 勾選下面這個選項則捨棄之前的進度與結果，從頭重新處理。
從頭重新處理 = False  #@param {type:"boolean"}

from IPython.display import Markdown, display
from google.colab import files

if not SELECTED or not Path(SELECTED).exists():
    raise SystemExit("⚠️ 請先在「步驟 2」選擇錄音檔")

meta = dict(title=會議名稱.strip() or Path(SELECTED).stem, date=會議日期, location=地點, chair=主席,
            num_speakers=None if 講者人數 == "自動判斷" else int(講者人數), hotwords=專有名詞.strip())
TR, MINUTES, META = mn.process(str(SELECTED), meta, 產生會議記錄, 從頭重新處理)

display(Markdown("---\n## 🔊 各講者聲音片段（用來辨認是誰，可到步驟 4 填上姓名）"))
mn.preview_speakers(TR, str(SELECTED))
if MINUTES:
    display(Markdown("---\n" + MINUTES))
if 下載Word到電腦 or not USE_DRIVE:
    for _f in mn.output_files(SELECTED, META):
        files.download(str(_f))

In [ ]:
#@title 步驟 4（選用）：替講者填上真實姓名
#@markdown 參考步驟 3 的「各講者聲音片段」，填入姓名（不需要的留空）。逐字稿與會議記錄都會更新。
#@markdown 要修改**較早處理過**的會議：先在步驟 2 選擇那個錄音（標示 ✅已處理），再執行本步驟。
講者_1 = ""  #@param {type:"string"}
講者_2 = ""  #@param {type:"string"}
講者_3 = ""  #@param {type:"string"}
講者_4 = ""  #@param {type:"string"}
講者_5 = ""  #@param {type:"string"}
講者_6 = ""  #@param {type:"string"}
講者_7 = ""  #@param {type:"string"}
講者_8 = ""  #@param {type:"string"}
#@markdown 勾選後會依新姓名**重新撰寫**會議記錄（約 2～5 分鐘）；不勾選只替換名稱（立即完成）。
重新產生會議記錄 = False  #@param {type:"boolean"}
下載Word到電腦 = False  #@param {type:"boolean"}

from IPython.display import Markdown, display
from google.colab import files

if not SELECTED:
    raise SystemExit("⚠️ 請先在「步驟 2」選擇錄音檔")
_names = [講者_1, 講者_2, 講者_3, 講者_4, 講者_5, 講者_6, 講者_7, 講者_8]
_mapping = {f"講者 {i}": n for i, n in enumerate(_names, 1) if n.strip()}
TR, MINUTES, META = mn.rename(SELECTED, _mapping, 重新產生會議記錄)
if MINUTES:
    display(Markdown("---\n" + MINUTES))
if 下載Word到電腦 or not USE_DRIVE:
    for _f in mn.output_files(SELECTED, META):
        files.download(str(_f))

---
### ❓ 常見問題
- **出現「無法連線到 GPU 後端」或「已達使用上限」**：免費 GPU 額度暫時用完，請過幾小時或隔天再試；也可以不用 GPU 執行（較慢）。
- **講者分得太多或太少**：在步驟 2 選擇正確的「講者人數」再執行一次。
- **專有名詞辨識錯誤**：在步驟 2 的「專有名詞」欄填入正確寫法。
- **結果檔案在哪裡**：和錄音檔同一個資料夾，檔名以會議名稱開頭：`會議記錄.docx`／`.md`、`逐字稿.docx`／`.txt`／`.srt`，另有一個 `_處理資料.json`（供改講者姓名用，請勿刪除）。
- **雲端硬碟授權失敗（credential propagation was unsuccessful）**：重新執行步驟 1，授權頁面點「全部選取」勾選所有權限；或確認瀏覽器沒有封鎖彈出式視窗。
- **資料夾清單沒看到剛放進去的檔案**：按清單上方的「🔄 重新整理」。
- **處理到一半中斷**：講者分離完成時、語音辨識每 20 秒、每段摘要完成時，進度都會自動存到雲端硬碟。重新執行步驟 1，在步驟 2 選同一個錄音（標示 ⏸️處理到一半），再執行步驟 3，就會從中斷處繼續。
- **處理很慢**：先看步驟 1 是否顯示「✅ 語音辨識使用 GPU」；處理完成時會顯示各階段耗時，可截圖回報以便判斷瓶頸。

---
## 🙏 致謝與授權（Credits）

本系統使用以下開源模型與軟體。模型都從 **[Hugging Face](https://huggingface.co)** 下載。謹向各模型的作者、轉換者與開源社群致謝。

### AI 模型

| 用途 | 模型（Hugging Face） | 原始作者／來源 | 授權 |
|---|---|---|---|
| 語音辨識 | [MediaTek-Research/Breeze-ASR-25](https://huggingface.co/MediaTek-Research/Breeze-ASR-25) | 聯發創新基地（MediaTek Research）。以 [openai/whisper-large-v2](https://huggingface.co/openai/whisper-large-v2) 為基礎微調 | Apache-2.0 |
| 語音辨識（加速格式） | [SoybeanMilk/faster-whisper-Breeze-ASR-25](https://huggingface.co/SoybeanMilk/faster-whisper-Breeze-ASR-25) | SoybeanMilk：轉換為 CTranslate2 格式 | Apache-2.0 |
| 講者分段 | [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) | pyannote.audio（Hervé Bredin 等，CNRS） | MIT |
| 講者分段（ONNX） | [csukuangfj/sherpa-onnx-pyannote-segmentation-3-0](https://huggingface.co/csukuangfj/sherpa-onnx-pyannote-segmentation-3-0) | Fangjun Kuang（k2-fsa／sherpa-onnx）：轉換為 ONNX | MIT（同原模型） |
| 講者聲紋 | [csukuangfj/speaker-embedding-models](https://huggingface.co/csukuangfj/speaker-embedding-models)（`3dspeaker_speech_eres2netv2_sv_zh-cn_16k-common.onnx`） | 原模型：阿里巴巴通義實驗室 [3D-Speaker](https://github.com/modelscope/3D-Speaker) ERes2NetV2；Fangjun Kuang 轉換為 ONNX | 依原模型授權 |
| 標點符號 | [csukuangfj/sherpa-onnx-punct-ct-transformer-zh-en-vocab272727-2024-04-12](https://huggingface.co/csukuangfj/sherpa-onnx-punct-ct-transformer-zh-en-vocab272727-2024-04-12) | 原模型：阿里巴巴達摩院 [FunASR](https://github.com/modelscope/FunASR) CT-Transformer（[ModelScope](https://modelscope.cn/models/iic/punc_ct-transformer_zh-cn-common-vocab272727-pytorch/summary)）；Fangjun Kuang 轉換為 ONNX | 依原模型授權 |
| 會議摘要（Colab） | [Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507)，或沒有 GPU 時用 [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) | 阿里巴巴通義千問 Qwen 團隊 | Apache-2.0 |
| 會議摘要（本機版） | [unsloth/Qwen3-4B-Instruct-2507-GGUF](https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF) | Qwen 團隊；Unsloth 量化為 GGUF 格式 | Apache-2.0 |

### 開源軟體

| 軟體 | 用途 | 授權 |
|---|---|---|
| [faster-whisper](https://github.com/SYSTRAN/faster-whisper)／[CTranslate2](https://github.com/OpenNMT/CTranslate2) | 語音辨識推論 | MIT |
| [sherpa-onnx](https://github.com/k2-fsa/sherpa-onnx)（k2-fsa） | 講者分離、標點還原 | Apache-2.0 |
| [Hugging Face Transformers](https://github.com/huggingface/transformers)／[huggingface_hub](https://github.com/huggingface/huggingface_hub) | 摘要模型推論、模型下載 | Apache-2.0 |
| [llama.cpp](https://github.com/ggml-org/llama.cpp)／[llama-cpp-python](https://github.com/abetlen/llama-cpp-python) | 本機版摘要模型推論 | MIT |
| [OpenCC](https://github.com/BYVoid/OpenCC)（opencc-python-reimplemented） | 轉為台灣正體與台灣慣用詞 | Apache-2.0 |
| [python-docx](https://github.com/python-openxml/python-docx) | 產生 Word 檔 | MIT |
| [FFmpeg](https://ffmpeg.org) | 音訊格式轉換 | LGPL／GPL |

### 引用（Citation）

使用 Breeze-ASR-25 的研究或發表，請依作者要求引用：

```bibtex
@article{chou2025selfrefiningframeworkenhancingasr,
  title={A Self-Refining Framework for Enhancing ASR Using TTS-Synthesized Data},
  author={Cheng Kang Chou and Chan-Jan Hsu and Ho-Lam Chung and Liang-Hsuan Tseng and Hsi-Chun Cheng and Yu-Kuan Fu and Kuan Po Huang and Hung-Yi Lee},
  journal={arXiv preprint arXiv:2506.11130},
  year={2025}
}
```

> 各模型的完整授權條款請以其 Hugging Face 或原始專案頁面為準。標示「依原模型授權」的模型，請至原始頁面（3D-Speaker、FunASR／ModelScope）確認授權內容。